In [ ]:
from pathlib import Path

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "python").is_dir() and (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from the "
        "credit-risk-analytics project folder or one of its subfolders."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
LOCAL_DATA_DIR = PROJECT_ROOT / "local_data" / "processed"
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "local_outputs"

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_DIR)
print("Local processed-data folder:", LOCAL_DATA_DIR)
print("Local output folder:", LOCAL_OUTPUT_DIR)


In [ ]:
# ============================================================
# NOTEBOOK 4 — REBUILD MODELLING ENVIRONMENT
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix
)

import joblib
import os

print("=" * 70)
print("LOADING HOME CREDIT DATA")
print("=" * 70)

# Load the training data from the documented local data folder.
data_path = DATA_DIR / "application_train.csv"

df = pd.read_csv(data_path)

print(f"\nDataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

print("\nTarget distribution:")
print(df["TARGET"].value_counts())

print("\nTarget proportions:")
print(df["TARGET"].value_counts(normalize=True))

In [ ]:
# ============================================================
# CELL 2 — FEATURE ENGINEERING
# ============================================================

print("=" * 70)
print("CREATING CREDIT RISK FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Target
# ------------------------------------------------------------

y = df["TARGET"].copy()

# ------------------------------------------------------------
# 2. Age
# Home Credit stores age as negative days
# ------------------------------------------------------------

df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365.25)

# ------------------------------------------------------------
# 3. Employment duration
# Home Credit uses 365243 as an anomalous placeholder
# ------------------------------------------------------------

df["DAYS_EMPLOYED_ANOMALY"] = (
    df["DAYS_EMPLOYED"] == 365243
).astype(int)

df["YEARS_EMPLOYED"] = (
    df["DAYS_EMPLOYED"]
    .replace(365243, np.nan)
    .abs() / 365.25
)

# ------------------------------------------------------------
# 4. Financial ratios
# ------------------------------------------------------------

df["CREDIT_TO_INCOME_RATIO"] = (
    df["AMT_CREDIT"] /
    df["AMT_INCOME_TOTAL"].replace(0, np.nan)
)

df["ANNUITY_TO_INCOME_RATIO"] = (
    df["AMT_ANNUITY"] /
    df["AMT_INCOME_TOTAL"].replace(0, np.nan)
)

df["GOODS_TO_CREDIT_RATIO"] = (
    df["AMT_GOODS_PRICE"] /
    df["AMT_CREDIT"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 5. Missingness indicators
# ------------------------------------------------------------

df["EXT_SOURCE_1_MISSING"] = (
    df["EXT_SOURCE_1"].isna()
).astype(int)

df["EXT_SOURCE_2_MISSING"] = (
    df["EXT_SOURCE_2"].isna()
).astype(int)

df["EXT_SOURCE_3_MISSING"] = (
    df["EXT_SOURCE_3"].isna()
).astype(int)

df["YEARS_EMPLOYED_MISSING"] = (
    df["YEARS_EMPLOYED"].isna()
).astype(int)

df["OCCUPATION_TYPE_MISSING"] = (
    df["OCCUPATION_TYPE"].isna()
).astype(int)

# ------------------------------------------------------------
# 6. Define features
# ------------------------------------------------------------

numeric_features = [
    "AGE_YEARS",
    "YEARS_EMPLOYED",
    "DAYS_EMPLOYED_ANOMALY",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CREDIT_TO_INCOME_RATIO",
    "ANNUITY_TO_INCOME_RATIO",
    "GOODS_TO_CREDIT_RATIO",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1_MISSING",
    "EXT_SOURCE_2_MISSING",
    "EXT_SOURCE_3_MISSING",
    "YEARS_EMPLOYED_MISSING",
    "OCCUPATION_TYPE_MISSING",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED"
]

categorical_features = [
    "CODE_GENDER",
    "NAME_FAMILY_STATUS",
    "NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    "NAME_CONTRACT_TYPE",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_HOUSING_TYPE"
]

features = numeric_features + categorical_features

X = df[features].copy()

print("\n✓ Feature engineering completed.")

print(f"\nNumber of numerical features: {len(numeric_features)}")
print(f"Number of categorical features: {len(categorical_features)}")
print(f"Total modelling features: {len(features)}")

print("\nEngineered features:")
print([
    "AGE_YEARS",
    "YEARS_EMPLOYED",
    "DAYS_EMPLOYED_ANOMALY",
    "CREDIT_TO_INCOME_RATIO",
    "ANNUITY_TO_INCOME_RATIO",
    "GOODS_TO_CREDIT_RATIO",
    "EXT_SOURCE_1_MISSING",
    "EXT_SOURCE_2_MISSING",
    "EXT_SOURCE_3_MISSING",
    "YEARS_EMPLOYED_MISSING",
    "OCCUPATION_TYPE_MISSING"
])

In [ ]:
print(df.shape)

In [ ]:
print("Dataset shape:", df.shape)

print("\nNumber of modelling features:", len(features))

print("\nMissing modelling features:")
missing_features = [col for col in features if col not in df.columns]

if missing_features:
    print(missing_features)
else:
    print("None ✓")

In [ ]:
df
y
numeric_features
categorical_features
features
X

In [ ]:
# ============================================================
# CELL 3 — TRAIN / VALIDATION SPLIT + PREPROCESSING
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Train / validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TRAIN / VALIDATION SPLIT")
print("=" * 70)

print(f"Training observations:   {X_train.shape[0]:,}")
print(f"Validation observations: {X_valid.shape[0]:,}")

print("\nTraining default rate:")
print(y_train.mean())

print("\nValidation default rate:")
print(y_valid.mean())


# ------------------------------------------------------------
# Numerical preprocessing
# ------------------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


# ------------------------------------------------------------
# Categorical preprocessing
# ------------------------------------------------------------

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)


# ------------------------------------------------------------
# Combined preprocessing
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("\n✓ Preprocessing pipeline created.")

In [ ]:
# ============================================================
# CELL 4 — TRAIN THREE CLASSIFICATION MODELS
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

print("=" * 70)
print("TRAINING CLASSIFICATION MODELS")
print("=" * 70)


# ============================================================
# 1. LOGISTIC REGRESSION
# ============================================================

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

print("\nTraining Logistic Regression...")
logistic_model.fit(X_train, y_train)

y_pred_proba = logistic_model.predict_proba(X_valid)[:, 1]

print("✓ Logistic Regression complete.")


# ============================================================
# 2. RANDOM FOREST
# ============================================================

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

print("\nTraining Random Forest...")
random_forest_model.fit(X_train, y_train)

rf_pred_proba = random_forest_model.predict_proba(X_valid)[:, 1]

print("✓ Random Forest complete.")


# ============================================================
# 3. GRADIENT BOOSTING
# ============================================================

gradient_boosting_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ]
)

print("\nTraining Gradient Boosting...")
gradient_boosting_model.fit(X_train, y_train)

gb_pred_proba = gradient_boosting_model.predict_proba(X_valid)[:, 1]

print("✓ Gradient Boosting complete.")

print("\n" + "=" * 70)
print("ALL THREE MODELS TRAINED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 5 — MODEL EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Default 0.50 predictions
logistic_pred = (y_pred_proba >= 0.50).astype(int)
rf_pred = (rf_pred_proba >= 0.50).astype(int)
gb_pred = (gb_pred_proba >= 0.50).astype(int)


# ------------------------------------------------------------
# Logistic Regression
# ------------------------------------------------------------

logistic_accuracy = accuracy_score(y_valid, logistic_pred)
logistic_precision = precision_score(y_valid, logistic_pred, zero_division=0)
logistic_recall = recall_score(y_valid, logistic_pred, zero_division=0)
logistic_f1 = f1_score(y_valid, logistic_pred, zero_division=0)
logistic_roc_auc = roc_auc_score(y_valid, y_pred_proba)
logistic_pr_auc = average_precision_score(y_valid, y_pred_proba)
logistic_brier = brier_score_loss(y_valid, y_pred_proba)


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

rf_accuracy = accuracy_score(y_valid, rf_pred)
rf_precision = precision_score(y_valid, rf_pred, zero_division=0)
rf_recall = recall_score(y_valid, rf_pred, zero_division=0)
rf_f1 = f1_score(y_valid, rf_pred, zero_division=0)
rf_roc_auc = roc_auc_score(y_valid, rf_pred_proba)
rf_pr_auc = average_precision_score(y_valid, rf_pred_proba)
rf_brier = brier_score_loss(y_valid, rf_pred_proba)


# ------------------------------------------------------------
# Gradient Boosting
# ------------------------------------------------------------

gb_accuracy = accuracy_score(y_valid, gb_pred)
gb_precision = precision_score(y_valid, gb_pred, zero_division=0)
gb_recall = recall_score(y_valid, gb_pred, zero_division=0)
gb_f1 = f1_score(y_valid, gb_pred, zero_division=0)
gb_roc_auc = roc_auc_score(y_valid, gb_pred_proba)
gb_pr_auc = average_precision_score(y_valid, gb_pred_proba)
gb_brier = brier_score_loss(y_valid, gb_pred_proba)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("MODEL PERFORMANCE — DEFAULT THRESHOLD = 0.50")
print("=" * 80)

print(f"\n{'Metric':<20} {'Logistic':>15} {'Random Forest':>18} {'Gradient Boosting':>20}")
print("-" * 80)

print(f"{'Accuracy':<20} {logistic_accuracy:>15.4f} {rf_accuracy:>18.4f} {gb_accuracy:>20.4f}")
print(f"{'Precision':<20} {logistic_precision:>15.4f} {rf_precision:>18.4f} {gb_precision:>20.4f}")
print(f"{'Recall':<20} {logistic_recall:>15.4f} {rf_recall:>18.4f} {gb_recall:>20.4f}")
print(f"{'F1':<20} {logistic_f1:>15.4f} {rf_f1:>18.4f} {gb_f1:>20.4f}")
print(f"{'ROC-AUC':<20} {logistic_roc_auc:>15.4f} {rf_roc_auc:>18.4f} {gb_roc_auc:>20.4f}")
print(f"{'PR-AUC':<20} {logistic_pr_auc:>15.4f} {rf_pr_auc:>18.4f} {gb_pr_auc:>20.4f}")
print(f"{'Brier Score':<20} {logistic_brier:>15.4f} {rf_brier:>18.4f} {gb_brier:>20.4f}")

In [ ]:
# ============================================================
# CELL 6 — SAVE FINAL MODEL
# ============================================================

joblib.dump(
    gradient_boosting_model,
    LOCAL_OUTPUT_DIR / "gradient_boosting_final_model.joblib"
)

print("✓ Gradient Boosting final model saved.")
print(os.path.abspath(LOCAL_OUTPUT_DIR / "gradient_boosting_final_model.joblib"))

In [ ]:
# ============================================================
# CELL 6 — GRADIENT BOOSTING THRESHOLD ANALYSIS
# ============================================================

thresholds = np.arange(0.05, 0.66, 0.05)

threshold_results = []

for threshold in thresholds:

    predictions = (gb_pred_proba >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_valid, predictions),
        "Precision": precision_score(
            y_valid, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_valid, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_valid, predictions, zero_division=0
        )
    })

threshold_plot_df = pd.DataFrame(threshold_results)

print("=" * 80)
print("GRADIENT BOOSTING — THRESHOLD ANALYSIS")
print("=" * 80)

display(
    threshold_plot_df.round(4)
)

In [ ]:
# ============================================================
# CELL 7 — SELECT BEST F1 THRESHOLD
# ============================================================

best_f1_row = threshold_plot_df.loc[
    threshold_plot_df["F1"].idxmax()
]

gb_threshold = best_f1_row["Threshold"]

gb_optimal_pred = (
    gb_pred_proba >= gb_threshold
).astype(int)

print("=" * 70)
print("SELECTED GRADIENT BOOSTING THRESHOLD")
print("=" * 70)

print(f"\nSelected threshold: {gb_threshold:.2f}")
print(f"Precision:          {precision_score(y_valid, gb_optimal_pred, zero_division=0):.4f}")
print(f"Recall:             {recall_score(y_valid, gb_optimal_pred, zero_division=0):.4f}")
print(f"F1:                 {f1_score(y_valid, gb_optimal_pred, zero_division=0):.4f}")
print(f"Accuracy:           {accuracy_score(y_valid, gb_optimal_pred):.4f}")

In [ ]:
# ============================================================
# CELL 8 — FINAL GRADIENT BOOSTING CONFUSION MATRIX
# ============================================================

gb_final_cm = confusion_matrix(
    y_valid,
    gb_optimal_pred
)

tn, fp, fn, tp = gb_final_cm.ravel()

print("=" * 70)
print("FINAL GRADIENT BOOSTING — CONFUSION MATRIX")
print("=" * 70)

print("\nConfusion Matrix:")
print(gb_final_cm)

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

print("\nFinal Threshold:", gb_threshold)

print("\nFinal Performance:")
print(f"Accuracy : {accuracy_score(y_valid, gb_optimal_pred):.4f}")
print(f"Precision: {precision_score(y_valid, gb_optimal_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_valid, gb_optimal_pred, zero_division=0):.4f}")
print(f"F1       : {f1_score(y_valid, gb_optimal_pred, zero_division=0):.4f}")

In [ ]:
# ============================================================
# CELL 9 — FINAL MODEL COMPARISON
# ============================================================

# Logistic Regression at its F1-selected threshold
logistic_threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.05):
    predictions = (y_pred_proba >= threshold).astype(int)

    logistic_threshold_results.append({
        "Threshold": threshold,
        "F1": f1_score(y_valid, predictions, zero_division=0)
    })

logistic_threshold_df = pd.DataFrame(
    logistic_threshold_results
)

logistic_threshold = logistic_threshold_df.loc[
    logistic_threshold_df["F1"].idxmax(),
    "Threshold"
]

logistic_final_pred = (
    y_pred_proba >= logistic_threshold
).astype(int)


# Random Forest at its F1-selected threshold
rf_threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.05):
    predictions = (rf_pred_proba >= threshold).astype(int)

    rf_threshold_results.append({
        "Threshold": threshold,
        "F1": f1_score(y_valid, predictions, zero_division=0)
    })

rf_threshold_df = pd.DataFrame(
    rf_threshold_results
)

rf_threshold = rf_threshold_df.loc[
    rf_threshold_df["F1"].idxmax(),
    "Threshold"
]

rf_final_pred = (
    rf_pred_proba >= rf_threshold
).astype(int)


# ------------------------------------------------------------
# Build comparison dynamically
# ------------------------------------------------------------

final_model_comparison = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],

    "Threshold": [
        logistic_threshold,
        rf_threshold,
        gb_threshold
    ],

    "Accuracy": [
        accuracy_score(y_valid, logistic_final_pred),
        accuracy_score(y_valid, rf_final_pred),
        accuracy_score(y_valid, gb_optimal_pred)
    ],

    "Precision": [
        precision_score(y_valid, logistic_final_pred, zero_division=0),
        precision_score(y_valid, rf_final_pred, zero_division=0),
        precision_score(y_valid, gb_optimal_pred, zero_division=0)
    ],

    "Recall": [
        recall_score(y_valid, logistic_final_pred, zero_division=0),
        recall_score(y_valid, rf_final_pred, zero_division=0),
        recall_score(y_valid, gb_optimal_pred, zero_division=0)
    ],

    "F1": [
        f1_score(y_valid, logistic_final_pred, zero_division=0),
        f1_score(y_valid, rf_final_pred, zero_division=0),
        f1_score(y_valid, gb_optimal_pred, zero_division=0)
    ],

    "ROC_AUC": [
        roc_auc_score(y_valid, y_pred_proba),
        roc_auc_score(y_valid, rf_pred_proba),
        roc_auc_score(y_valid, gb_pred_proba)
    ],

    "PR_AUC": [
        average_precision_score(y_valid, y_pred_proba),
        average_precision_score(y_valid, rf_pred_proba),
        average_precision_score(y_valid, gb_pred_proba)
    ],

    "Brier_Score": [
        brier_score_loss(y_valid, y_pred_proba),
        brier_score_loss(y_valid, rf_pred_proba),
        brier_score_loss(y_valid, gb_pred_proba)
    ]
})

print("=" * 100)
print("FINAL MODEL COMPARISON")
print("=" * 100)

display(
    final_model_comparison.round(4)
)

In [ ]:
# ============================================================
# CELL 10 — SAVE FINAL RESULTS
# ============================================================

final_model_comparison.to_csv(LOCAL_OUTPUT_DIR / "final_model_comparison.csv",
    index=False
)

threshold_plot_df.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_threshold_analysis.csv",
    index=False
)

joblib.dump(
    gradient_boosting_model,
    LOCAL_OUTPUT_DIR / "gradient_boosting_final_model.joblib"
)

print("✓ Final model comparison saved.")
print("✓ Threshold analysis saved.")
print("✓ Final Gradient Boosting model saved.")

print("\nFiles saved in:")
print(os.getcwd())

In [ ]:
# ============================================================
# CELL 11 — GRADIENT BOOSTING FEATURE IMPORTANCE
# ============================================================

print("=" * 80)
print("GRADIENT BOOSTING — FEATURE IMPORTANCE")
print("=" * 80)

# Extract fitted preprocessing and classifier
gb_preprocessor = gradient_boosting_model.named_steps["preprocessor"]
gb_classifier = gradient_boosting_model.named_steps["model"]

# Get transformed feature names
gb_feature_names = gb_preprocessor.get_feature_names_out()

# Get feature importance values
gb_importance_values = gb_classifier.feature_importances_

# Create feature importance dataframe
gb_feature_importance = pd.DataFrame({
    "Feature": gb_feature_names,
    "Importance": gb_importance_values
})

# Remove preprocessing prefixes for easier interpretation
gb_feature_importance["Feature"] = (
    gb_feature_importance["Feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

# Sort from most to least important
gb_feature_importance = (
    gb_feature_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 20 features:")
display(
    gb_feature_importance.head(20).round(6)
)

In [ ]:
# ============================================================
# CELL 12 — TOP 15 GRADIENT BOOSTING FEATURES
# ============================================================

import matplotlib.pyplot as plt

top_features = gb_feature_importance.head(15).sort_values(
    "Importance"
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 15 Gradient Boosting Features")

plt.tight_layout()
plt.show()

In [ ]:
gb_feature_importance.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_feature_importance.csv",
    index=False
)

print("✓ Feature importance saved.")

In [ ]:
# ============================================================
# CELL 13 — LOGISTIC REGRESSION COEFFICIENTS & ODDS RATIOS
# ============================================================

print("=" * 80)
print("LOGISTIC REGRESSION — ODDS RATIOS")
print("=" * 80)

# Extract fitted logistic regression components
logistic_preprocessor = logistic_model.named_steps["preprocessor"]
logistic_classifier = logistic_model.named_steps["classifier"]

# Get transformed feature names
logistic_feature_names = (
    logistic_preprocessor.get_feature_names_out()
)

# Get coefficients
logistic_coefficients = (
    logistic_classifier.coef_[0]
)

# Create results dataframe
logistic_odds_ratios = pd.DataFrame({
    "Feature": logistic_feature_names,
    "Coefficient": logistic_coefficients,
    "Odds_Ratio": np.exp(logistic_coefficients)
})

# Clean feature names
logistic_odds_ratios["Feature"] = (
    logistic_odds_ratios["Feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

# Absolute coefficient for sorting
logistic_odds_ratios["Abs_Coefficient"] = (
    logistic_odds_ratios["Coefficient"].abs()
)

# Sort by absolute coefficient
logistic_odds_ratios = (
    logistic_odds_ratios
    .sort_values("Abs_Coefficient", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 20 Logistic Regression effects:")
display(
    logistic_odds_ratios.head(20).round(4)
)

In [ ]:
# ============================================================
# CELL 14 — STRONGEST POSITIVE AND NEGATIVE ASSOCIATIONS
# ============================================================

positive_effects = (
    logistic_odds_ratios
    .sort_values("Coefficient", ascending=False)
    .head(10)
)

negative_effects = (
    logistic_odds_ratios
    .sort_values("Coefficient", ascending=True)
    .head(10)
)

print("=" * 80)
print("STRONGEST POSITIVE ASSOCIATIONS WITH DEFAULT ODDS")
print("=" * 80)

display(
    positive_effects[
        ["Feature", "Coefficient", "Odds_Ratio"]
    ].round(4)
)

print("=" * 80)
print("STRONGEST NEGATIVE ASSOCIATIONS WITH DEFAULT ODDS")
print("=" * 80)

display(
    negative_effects[
        ["Feature", "Coefficient", "Odds_Ratio"]
    ].round(4)
)

In [ ]:
# ============================================================
# CELL 17 — MODEL CALIBRATION
# ============================================================

from sklearn.calibration import calibration_curve

# Calculate calibration curves
logistic_fraction, logistic_mean = calibration_curve(
    y_valid,
    y_pred_proba,
    n_bins=10,
    strategy="quantile"
)

rf_fraction, rf_mean = calibration_curve(
    y_valid,
    rf_pred_proba,
    n_bins=10,
    strategy="quantile"
)

gb_fraction, gb_mean = calibration_curve(
    y_valid,
    gb_pred_proba,
    n_bins=10,
    strategy="quantile"
)

print("=" * 80)
print("MODEL CALIBRATION")
print("=" * 80)

calibration_results = pd.DataFrame({
    "Model": (
        ["Logistic Regression"] * len(logistic_fraction)
        + ["Random Forest"] * len(rf_fraction)
        + ["Gradient Boosting"] * len(gb_fraction)
    ),
    "Mean_Predicted_Risk": np.concatenate([
        logistic_mean,
        rf_mean,
        gb_mean
    ]),
    "Observed_Default_Rate": np.concatenate([
        logistic_fraction,
        rf_fraction,
        gb_fraction
    ])
})

display(
    calibration_results.round(4)
)

In [ ]:
# ============================================================
# CELL 18 — CALIBRATION PLOT
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 7))

plt.plot(
    logistic_mean,
    logistic_fraction,
    marker="o",
    label="Logistic Regression"
)

plt.plot(
    rf_mean,
    rf_fraction,
    marker="o",
    label="Random Forest"
)

plt.plot(
    gb_mean,
    gb_fraction,
    marker="o",
    label="Gradient Boosting"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Default Rate")
plt.title("Model Calibration")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 19 — FINAL GRADIENT BOOSTING ERROR ANALYSIS
# ============================================================

gb_error_analysis = X_valid.copy()

gb_error_analysis["Actual_Default"] = y_valid.values
gb_error_analysis["Predicted_Default"] = gb_optimal_pred
gb_error_analysis["Predicted_Risk"] = gb_pred_proba

gb_error_analysis["Error_Type"] = np.select(
    [
        (gb_error_analysis["Actual_Default"] == 1) &
        (gb_error_analysis["Predicted_Default"] == 1),

        (gb_error_analysis["Actual_Default"] == 0) &
        (gb_error_analysis["Predicted_Default"] == 0),

        (gb_error_analysis["Actual_Default"] == 1) &
        (gb_error_analysis["Predicted_Default"] == 0),

        (gb_error_analysis["Actual_Default"] == 0) &
        (gb_error_analysis["Predicted_Default"] == 1)
    ],
    [
        "True Positive",
        "True Negative",
        "False Negative",
        "False Positive"
    ],
    default="Other"
)

error_type_summary = (
    gb_error_analysis["Error_Type"]
    .value_counts()
    .rename_axis("Error_Type")
    .reset_index(name="Count")
)

error_type_summary["Percentage"] = (
    error_type_summary["Count"] /
    len(gb_error_analysis) * 100
)

print("=" * 80)
print("GRADIENT BOOSTING — ERROR ANALYSIS")
print("=" * 80)

display(
    error_type_summary.round(2)
)

In [ ]:
gb_error_analysis.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_error_analysis.csv",
    index=False
)

error_type_summary.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_error_summary.csv",
    index=False
)

calibration_results.to_csv(LOCAL_OUTPUT_DIR / "model_calibration_results.csv",
    index=False
)

print("✓ Calibration results saved.")
print("✓ Error analysis saved.")

In [ ]:
# ============================================================
# CELL 20 — FINAL PROJECT RESULTS SUMMARY
# ============================================================

print("=" * 90)
print("FINAL CREDIT RISK MODEL — PROJECT SUMMARY")
print("=" * 90)

print("\nDATASET")
print("-" * 90)
print(f"Total observations:       {len(df):,}")
print(f"Training observations:     {len(X_train):,}")
print(f"Validation observations:  {len(X_valid):,}")
print(f"Overall default rate:     {y.mean():.2%}")

print("\nFINAL MODEL")
print("-" * 90)
print("Selected model:            Gradient Boosting")
print(f"Decision threshold:        {gb_threshold:.2f}")

print("\nMODEL DISCRIMINATION")
print("-" * 90)
print(f"ROC-AUC:                   {gb_roc_auc:.4f}")
print(f"PR-AUC:                    {gb_pr_auc:.4f}")

print("\nMODEL CALIBRATION")
print("-" * 90)
print(f"Brier Score:               {gb_brier:.4f}")

print("\nTHRESHOLD-BASED PERFORMANCE")
print("-" * 90)
print(f"Accuracy:                  {accuracy_score(y_valid, gb_optimal_pred):.4f}")
print(f"Precision:                 {precision_score(y_valid, gb_optimal_pred, zero_division=0):.4f}")
print(f"Recall:                    {recall_score(y_valid, gb_optimal_pred, zero_division=0):.4f}")
print(f"F1 Score:                  {f1_score(y_valid, gb_optimal_pred, zero_division=0):.4f}")

print("\nCONFUSION MATRIX")
print("-" * 90)
print(f"True Negatives:            {tn:,}")
print(f"False Positives:           {fp:,}")
print(f"False Negatives:           {fn:,}")
print(f"True Positives:            {tp:,}")

print("\nTOP PREDICTIVE FEATURES")
print("-" * 90)

for i, row in gb_feature_importance.head(10).iterrows():
    print(
        f"{i + 1:2d}. "
        f"{row['Feature']:<45} "
        f"{row['Importance']:.4f}"
    )

print("\n" + "=" * 90)
print("PROJECT MODELLING STAGE COMPLETE")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 21 — SAVE FINAL PROJECT OUTPUTS
# ============================================================

# Final model
joblib.dump(
    gradient_boosting_model,
    LOCAL_OUTPUT_DIR / "gradient_boosting_final_model.joblib"
)

# Main results
final_model_comparison.to_csv(LOCAL_OUTPUT_DIR / "final_model_comparison.csv",
    index=False
)

# Feature importance
gb_feature_importance.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_feature_importance.csv",
    index=False
)

# Threshold analysis
threshold_plot_df.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_threshold_analysis.csv",
    index=False
)

# Error analysis
error_type_summary.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_error_summary.csv",
    index=False
)

# Calibration
calibration_results.to_csv(LOCAL_OUTPUT_DIR / "model_calibration_results.csv",
    index=False
)

# Logistic odds ratios
logistic_odds_ratios.to_csv(LOCAL_OUTPUT_DIR / "logistic_regression_odds_ratios.csv",
    index=False
)

print("=" * 80)
print("FINAL PROJECT OUTPUTS SAVED")
print("=" * 80)

print("\nFiles:")
print("✓ gradient_boosting_final_model.joblib")
print("✓ final_model_comparison.csv")
print("✓ gradient_boosting_feature_importance.csv")
print("✓ gradient_boosting_threshold_analysis.csv")
print("✓ gradient_boosting_error_summary.csv")
print("✓ model_calibration_results.csv")
print("✓ logistic_regression_odds_ratios.csv")

print("\nLocation:")
print(os.getcwd())

In [ ]:
# ============================================================
# POWER BI DATA PACKAGE — FIXED VERSION
# ============================================================

import pandas as pd
import os

# Create folder
powerbi_folder = LOCAL_OUTPUT_DIR / "powerbi_outputs"
os.makedirs(powerbi_folder, exist_ok=True)


# ------------------------------------------------------------
# 1. MODEL COMPARISON
# ------------------------------------------------------------

final_model_comparison.to_csv(
    os.path.join(powerbi_folder, "model_comparison.csv"),
    index=False
)


# ------------------------------------------------------------
# 2. GRADIENT BOOSTING FEATURE IMPORTANCE
# ------------------------------------------------------------

gb_feature_importance.to_csv(
    os.path.join(powerbi_folder, "feature_importance.csv"),
    index=False
)


# ------------------------------------------------------------
# 3. GRADIENT BOOSTING THRESHOLD ANALYSIS
# ------------------------------------------------------------

# threshold_results is a list of dictionaries
threshold_analysis = pd.DataFrame(threshold_results)

threshold_analysis.to_csv(
    os.path.join(powerbi_folder, "threshold_analysis.csv"),
    index=False
)


# ------------------------------------------------------------
# 4. ERROR ANALYSIS
# ------------------------------------------------------------

error_analysis = pd.DataFrame({
    "Outcome": [
        "True Negative",
        "False Positive",
        "False Negative",
        "True Positive"
    ],
    "Count": [
        tn,
        fp,
        fn,
        tp
    ]
})

error_analysis.to_csv(
    os.path.join(powerbi_folder, "error_analysis.csv"),
    index=False
)


# ------------------------------------------------------------
# 5. CALIBRATION RESULTS
# ------------------------------------------------------------

calibration_results.to_csv(
    os.path.join(powerbi_folder, "calibration_results.csv"),
    index=False
)


# ------------------------------------------------------------
# 6. MODEL SUMMARY
# ------------------------------------------------------------

model_summary = pd.DataFrame({
    "Metric": [
        "Total Applicants",
        "Default Rate",
        "Non-default Rate",
        "Selected Model",
        "Selected Threshold",
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Value": [
        len(df),
        y.mean(),
        1 - y.mean(),
        "Gradient Boosting",
        gb_threshold,
        gb_roc_auc,
        gb_pr_auc,
        gb_brier,
        accuracy_score(y_valid, gb_optimal_pred),
        precision_score(y_valid, gb_optimal_pred, zero_division=0),
        recall_score(y_valid, gb_optimal_pred, zero_division=0),
        f1_score(y_valid, gb_optimal_pred, zero_division=0)
    ]
})

model_summary.to_csv(
    os.path.join(powerbi_folder, "model_summary.csv"),
    index=False
)


# ------------------------------------------------------------
# 7. CHECK OUTPUTS
# ------------------------------------------------------------

print("POWER BI DATA PACKAGE CREATED SUCCESSFULLY")
print("=" * 55)

for file in sorted(os.listdir(powerbi_folder)):
    print("✓", file)

print("\nFolder location:")
print(os.path.abspath(powerbi_folder))